# ระดับ 3f — ablation ต่างกันจริงหรือไม่: paired test + ความแปรปรวนจาก seed

**Paper:** Guo et al., *Revisiting Face Forgery Detection: From Facial Representation to Forgery Detection*, IEEE TPAMI 2026

**คำถาม:** องค์ประกอบของ CFB (2 branch, `L_dec`, UFM, `L_EDU`) ช่วยเพิ่ม AUC บน dataset ที่ไม่เคยเห็นจริงหรือไม่
ใน 3e ผลของเรากลับด้านจาก Table VI แต่ต่างกันไม่เกิน 3 จุด และช่วงความเชื่อมั่นของแต่ละโมเดลซ้อนกัน

ความบังเอิญมีที่มา 2 แบบ notebook นี้ตรวจทั้งคู่:

| ส่วน | ตรวจความบังเอิญจาก | วิธี | ต้องใช้ |
|---|---|---|---|
| **1** | ชุดทดสอบมีจำกัด (Celeb-DF 518 วิดีโอ) | **Paired test** บนวิดีโอชุดเดียวกัน: DeLong test + paired bootstrap ของ ΔAUC · แก้ multiple comparison ด้วย Holm | score จาก 3e (CPU, < 2 นาที) |
| **2** | การฝึก (seed) | ประเมิน checkpoint seed ใหม่จาก 3d2 แล้วดู mean ± SD ข้าม seed | GPU T4 ×1 (ข้ามอัตโนมัติถ้าไม่มี checkpoint จาก 3d2) |

paired test มีความไวกว่าการดูว่าช่วงความเชื่อมั่นซ้อนกันหรือไม่ เพราะทุกโมเดลถูกทดสอบบนวิดีโอเดียวกัน ความยากง่ายของวิดีโอจึงหักล้างกัน

**ก่อนรัน:** Add Input:
- output ของ **3e_V2** (`results_level3e/scores_*.csv`) — จำเป็น
- (ส่วน 2) output ของ **3d2** (`ablation_seeds/`) ทุก account · **3a** (FF++ test) · **3c** (`cdf_crops`) · 3e รุ่นที่มี `wdf_crops/` · GPU T4 ×1 · Internet On

## 0. ตั้งค่า

In [ ]:
import os, sys, re, json, time, glob, random, shutil, tarfile, hashlib, subprocess, warnings, io
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # ไม่พิมพ์ progress bar ของการดาวน์โหลด 157 ไฟล์ (ทำให้ notebook ใหญ่เกิน 1 MB ตอน commit)
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR  = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR   = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"
INPUT_DIR = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("FFD_INPUT", str(WORK_DIR / "input")))
REPO_DIR  = TMP_DIR / "FFDBackbone"
DATA_ROOT = TMP_DIR / "data"                     # แตก tar ของระดับ 3a (เฉพาะ test)
WDF_DIR   = WORK_DIR / "wdf_crops"               # เก็บไว้ใน working เพื่อใช้ซ้ำรอบหน้า (Add Input)
WEIGHT_DIR = TMP_DIR / "weights"
OUT_DIR   = WORK_DIR / "results_level3f"
for d in [TMP_DIR, DATA_ROOT, WDF_DIR, WEIGHT_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/zhenglab/FFDBackbone.git"
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/1jFIpb4TftJiL82h69c3Roqy8uIzGQP4G"

# --- โมเดล ---
USE_AUTHOR_CKPT   = False                         # ดาวน์โหลด checkpoint "Ours" ของผู้เขียนมาเทียบ
AUTHOR_CKPT_PATH  = None                         # ถ้าอัปเป็น Kaggle Dataset ไว้แล้วให้ใส่ path
AUTHOR_HINTS      = ["ours", "final", "ddbf", "dbbf", "mainbranch"]
AUTHOR_EXCLUDE    = ["beit-1k-face-55w"]

# --- dataset ---
DATASETS_P2 = [d for d in os.environ.get("FFD_DATASETS", "FF++,CDF,WDF").split(",") if d]  # ชุดที่ใช้ในส่วน 2
WDF_REPO   = "xingjunm/WildDeepfake"
WDF_SPLITS = {"real_test": 0, "fake_test": 1}    # label: 0 = real, 1 = fake
WDF_REALIGN = True                               # True = RetinaFace + align ซ้ำแบบเดียวกับ FF++/CDF (หาไม่เจอ → ใช้ภาพเดิม)
WDF_MAX_FILES = None                             # จำกัดจำนวนไฟล์ tar ต่อ split (None = ทั้งหมด) — ใช้ทดสอบเร็ว ๆ

# --- preprocessing / inference (ค่าตาม repo) ---
N_FRAMES, IMG_SIZE = 8, 224
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
DET_THRES = 0.5
BATCH_SIZE, NUM_WORKERS, USE_AMP = 64, 2, True

# --- UTOM (Sec. III-D + Supplementary 2.1, 2.3) ---
UTOM_LAMBDA, UTOM_EPS = 0.02, 1e-7
N_BOOT, SEED = 1000, 42

PAPER_MAIN = {"FF++": 99.36, "CDF": 90.46}
PAPER_TABLE6 = {   # Table VI: CDF, DFDC, FFIW, Avg AUC, Avg ACC
    "fb_ce": (89.18, 84.02, 86.48, 86.56, 79.37), "fb_edu": (89.34, 84.19, 88.76, 87.43, 80.96),
    "cfb_base": (89.77, 83.67, 87.69, 87.04, 79.26), "cfb_dec": (90.31, 84.89, 89.21, 88.14, 79.42),
    "cfb_dec_ufm": (91.16, 84.54, 90.22, 88.64, 80.84), "cfb_full": (90.46, 84.90, 90.97, 88.78, 82.29),
}
PAPER_TABLE9_CDF = {   # Table IX, CFB + L_EDU, source = CDF → target CDF (ACC %)
    "tau=0.5": 83.20, "tau_ot": 83.78, "tau_ot^UTOM": 85.33}

random.seed(SEED)
print("ON_KAGGLE:", ON_KAGGLE)


# --- ส่วน 1: paired test ---
N_BOOT_PAIR = 2000
ALPHA = 0.05
# ลำดับเดียวกับ Table VI · Δ = (เพิ่มองค์ประกอบ) − (ก่อนเพิ่ม)
COMPS = [("fb_ce", "fb_edu", "+ L_EDU (1 branch)"),
         ("fb_ce", "cfb_base", "+ branch ที่ 2"),
         ("cfb_base", "cfb_dec", "+ L_dec"),
         ("cfb_dec", "cfb_dec_ufm", "+ UFM"),
         ("cfb_dec_ufm", "cfb_full", "+ L_EDU (2 branch)"),
         ("fb_ce", "cfb_full", "FB+CE → CFB ครบ")]
PAPER_CDF = {k: v[0] for k, v in PAPER_TABLE6.items()}

# --- ส่วน 2: checkpoint จาก 3d2 ---
SEED_CKPTS = sorted(INPUT_DIR.rglob("ablation_seeds/*_s*/best.pth")) if INPUT_DIR.exists() else []
DATASETS = DATASETS_P2 if SEED_CKPTS else []      # ไม่มี checkpoint ใหม่ → ส่วน 2 ไม่โหลดข้อมูลและไม่รันโมเดล
print("checkpoint seed ใหม่:", [p.parent.name for p in SEED_CKPTS] or "ไม่มี → รันเฉพาะส่วน 1")

## 0.1 ติดตั้ง + clone repo (ใช้ในส่วน 2)

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=False)
pip_install("gdown")
try: import huggingface_hub
except ImportError: pip_install("huggingface_hub")
try: import timm
except ImportError: pip_install("timm"); import timm

import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print({"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})

def find_repo_in_input():
    """หา FFDBackbone ที่ Add Input ไว้ (โหมดออฟไลน์) — ค้นแค่ 2 ชั้นแรก ไม่ไล่ทั้ง dataset"""
    if not INPUT_DIR.exists():
        return None
    marker = Path("models") / "lib" / "BEiT_v2" / "modeling_finetune.py"
    for d in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
        cands = [d] + sorted(p for p in d.iterdir() if p.is_dir())[:50]
        for c in cands:
            if (c / marker).exists():
                return c
    return None

def get_repo():
    """1) ที่ clone ไว้แล้ว  2) จาก /kaggle/input (ออฟไลน์)  3) git clone"""
    if (REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py").exists():
        return "cached"
    src = find_repo_in_input()
    if src is not None:
        print("พบ repo ใน Add Input (โหมดออฟไลน์):", src)
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
        return "input"
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone repo ไม่สำเร็จ:\n  " + (r.stderr or "").strip() + "\n\n"
            "ถ้าขึ้น 'Could not resolve host' แปลว่า Internet ปิดอยู่ — แก้ได้ 2 ทาง\n"
            "  (ก) เปิดเน็ต: แผงขวาของ Kaggle -> Notebook options -> Internet = On\n"
            "      (ต้องยืนยันเบอร์โทรที่ kaggle.com/settings ก่อน ถ้ายังไม่เคยทำ)\n"
            "  (ข) โหมดออฟไลน์: ดาวน์โหลด zip ของ repo จาก\n"
            "      https://github.com/zhenglab/FFDBackbone/archive/refs/heads/main.zip\n"
            "      -> อัปเป็น Kaggle Dataset -> Add Input  (cell นี้จะหาเจอเอง)")
    return "clone"

REPO_SOURCE = get_repo()
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "(จาก Add Input — ไม่มี .git)"

def patch_repo(repo: Path):
    n = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8"); n += 1
    return n

print("repo commit:", REPO_COMMIT, "| patched:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))

import importlib.util
def load_module_from_file(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); sys.modules[name] = m
    spec.loader.exec_module(m); return m

modeling_finetune = load_module_from_file("beitv2_modeling_finetune",
                                          REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py")
from face_process.lib.ct.detection import FaceDetector
from face_process import face_utils
from scipy.stats import rankdata, norm

## 1. Paired test บน score จาก 3e_V2

- **DeLong test** (DeLong et al., 1988; fast version ของ Sun & Xu, 2014): ทดสอบว่า AUC ของ 2 โมเดลที่ใช้ชุดทดสอบเดียวกันต่างกันหรือไม่
- **Paired bootstrap:** สุ่มวิดีโอใหม่ (แยก real / fake) 2,000 รอบ ใช้ index เดียวกันกับทุกโมเดล → ช่วงความเชื่อมั่น 95% ของ ΔAUC
- **Holm correction** ต่อ dataset (6 การเปรียบเทียบ) เพื่อคุม family-wise error ที่ α = 0.05
- **MDE<sub>80</sub>** = 2.8 × SD(ΔAUC) ≈ ความต่างที่เล็กที่สุดที่ชุดทดสอบนี้ตรวจพบได้ด้วย power 80%
- เทียบ checkpoint ผู้เขียนกับโมเดลของเราด้วย เพื่อยืนยันว่า test นี้ตรวจความต่างที่มีจริงได้

In [ ]:
def load_scores():
    """เลือกโฟลเดอร์ results_level3e ที่มีไฟล์ score มากที่สุด (กรณี Add Input หลายรุ่น)"""
    dirs = {}
    for f in (INPUT_DIR.rglob("scores_*_*.csv") if INPUT_DIR.exists() else []):
        if "results_level3f" in str(f): continue
        dirs.setdefault(f.parent, []).append(f)
    assert dirs, "ไม่พบ scores_*.csv — Add Input เป็น output ของระดับ 3e_V2"
    best = sorted(dirs, key=lambda d: (len(dirs[d]), str(d)))[-1]
    out = {}
    for f in dirs[best]:
        m = re.match(r"scores_(.+)_(FF\+\+|CDF|WDF)\.csv$", f.name)
        if m: out[(m.group(1), m.group(2))] = pd.read_csv(f)
    print("ใช้ score จาก:", best, "·", len(out), "ไฟล์")
    return out

S = load_scores()
MODELS_P1 = sorted({m for m, _ in S})
DS_P1 = [d for d in ["CDF", "WDF", "FF++"] if any(k[1] == d for k in S)]
print("โมเดล:", MODELS_P1); print("dataset:", DS_P1)
vname = lambda v: "ours_cfb_full" if v == "cfb_full" else f"ours_{v}"

In [ ]:
def auc_rank(y, s):
    r = rankdata(s); n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def delong(y, s1, s2):
    """fast DeLong (Sun & Xu 2014) · คืน (AUC1, AUC2, z, p) สำหรับ 2 score บนตัวอย่างชุดเดียวกัน"""
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0
    m, n = pos.sum(), neg.sum()
    aucs, v10, v01 = [], [], []
    for s in (np.asarray(s1, float), np.asarray(s2, float)):
        X, Y = s[pos], s[neg]
        tz = rankdata(np.r_[X, Y]); tx = rankdata(X); ty = rankdata(Y)
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        v10.append((tz[:m] - tx) / n); v01.append(1 - (tz[m:] - ty) / m)
    S10, S01 = np.cov(np.vstack(v10)), np.cov(np.vstack(v01))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / n
    z = (aucs[0] - aucs[1]) / np.sqrt(var) if var > 0 else 0.0
    return aucs[0], aucs[1], z, 2 * norm.sf(abs(z))

def aligned(ds, names):
    """รวม score ของหลายโมเดลบน uid เดียวกัน · ตรวจว่า label ตรงกัน"""
    base = None
    for nm in names:
        d = S[(nm, ds)][["uid", "label", "p_main"]].rename(columns={"p_main": nm, "label": f"y_{nm}"})
        base = d if base is None else base.merge(d, on="uid", how="inner")
    ys = base[[f"y_{nm}" for nm in names]].values
    assert (ys == ys[:, :1]).all(), "label ไม่ตรงกันระหว่างไฟล์ score"
    return base["label" if "label" in base else f"y_{names[0]}"].values.astype(int), base

def boot_idx(y, B, seed):
    rng = np.random.default_rng(seed); i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]
    return [np.r_[rng.choice(i0, len(i0)), rng.choice(i1, len(i1))] for _ in range(B)]

def holm(p):
    p = np.asarray(p); o = np.argsort(p); k = len(p); adj = np.empty(k); run = 0
    for r, i in enumerate(o):
        run = max(run, (k - r) * p[i]); adj[i] = min(1.0, run)
    return adj

# ตรวจ DeLong กับ sklearn
_y = np.r_[np.zeros(50), np.ones(70)].astype(int); _r = np.random.default_rng(0)
_a, _b = _r.random(120) + _y * .3, _r.random(120) + _y * .2
assert abs(delong(_y, _a, _b)[0] - roc_auc_score(_y, _a)) < 1e-12 and abs(auc_rank(_y, _a) - roc_auc_score(_y, _a)) < 1e-12
print("ตรวจสอบ: AUC จาก DeLong / rank = sklearn ✓")

In [ ]:
rows, BOOT = [], {}
for ds in DS_P1:
    need = sorted({vname(a) for a, b, _ in COMPS} | {vname(b) for a, b, _ in COMPS} | {"author"})
    have = [nm for nm in need if (nm, ds) in S]
    y, T = aligned(ds, have)
    idx = boot_idx(y, N_BOOT_PAIR, SEED)
    BA = {nm: np.array([auc_rank(y[i], T[nm].values[i]) for i in idx]) for nm in have}
    BOOT[ds] = BA
    pairs = [(vname(a), vname(b), lab, a, b) for a, b, lab in COMPS] + [("ours_cfb_full", "author", "ours → checkpoint ผู้เขียน (ตรวจ power)", None, None)]
    for A_, B_, lab, a, b in pairs:
        if A_ not in have or B_ not in have: continue
        aA, aB, z, p = delong(y, T[A_].values, T[B_].values)
        d = BA[B_] - BA[A_]
        rows.append({"dataset": ds, "comparison": lab, "A": A_, "B": B_, "n": len(y),
                     "AUC_A": 100 * aA, "AUC_B": 100 * aB, "ΔAUC": 100 * (aB - aA),
                     "CI_lo": 100 * np.percentile(d, 2.5), "CI_hi": 100 * np.percentile(d, 97.5),
                     "p_boot": min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())), "p_DeLong": p,
                     "MDE80": 100 * 2.8 * d.std(ddof=1),
                     "paper_ΔCDF": (PAPER_CDF[b] - PAPER_CDF[a]) if a else np.nan, "ablation": a is not None})
SIG = pd.DataFrame(rows)
SIG["p_Holm"] = np.nan
for ds in DS_P1:
    k = (SIG.dataset == ds) & SIG.ablation
    SIG.loc[k, "p_Holm"] = holm(SIG.loc[k, "p_DeLong"].values)
SIG["สรุป"] = np.where(SIG.ablation, np.where(SIG.p_Holm < ALPHA, "ต่างจริง", "ไม่ต่างอย่างมีนัยสำคัญ"),
                       np.where(SIG.p_DeLong < ALPHA, "ต่างจริง", "ไม่ต่าง"))
SIG.to_csv(OUT_DIR / "paired_tests.csv", index=False)
show = ["comparison", "AUC_A", "AUC_B", "ΔAUC", "CI_lo", "CI_hi", "p_DeLong", "p_Holm", "MDE80", "paper_ΔCDF", "สรุป"]
for ds in DS_P1:
    print(f"\n=== {ds} (n = {int(SIG[SIG.dataset == ds].n.iloc[0])}) · main branch · Δ = B − A ===")
    print(SIG[SIG.dataset == ds][show].round(4).to_string(index=False))

In [ ]:
INK2, GRID = "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.edgecolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
fig, axes = plt.subplots(1, len(DS_P1), figsize=(5.2 * len(DS_P1), 3.8), sharey=True)
LAB_EN = {"+ L_EDU (1 branch)": "+ L_EDU (1 branch)", "+ branch ที่ 2": "+ 2nd branch", "+ L_dec": "+ L_dec",
          "+ UFM": "+ UFM", "+ L_EDU (2 branch)": "+ L_EDU (2 branch)", "FB+CE → CFB ครบ": "FB+CE -> CFB full",
          "ours → checkpoint ผู้เขียน (ตรวจ power)": "ours -> author (power check)"}
for ax, ds in zip(np.atleast_1d(axes), DS_P1):
    d = SIG[SIG.dataset == ds].reset_index(drop=True)
    yy = np.arange(len(d))[::-1]
    for i, r in d.iterrows():
        c = "#7a1f22" if r["สรุป"] == "ต่างจริง" else "#8a8278"
        ax.plot([r.CI_lo, r.CI_hi], [yy[i]] * 2, color=c, lw=2); ax.plot(r["ΔAUC"], yy[i], "o", color=c)
        if not np.isnan(r["paper_ΔCDF"]) and ds == "CDF": ax.plot(r["paper_ΔCDF"], yy[i], "x", color="#2a78d6", ms=8)
    ax.axvline(0, color=INK2, lw=1); ax.set_yticks(yy); ax.set_yticklabels([LAB_EN.get(c, c) for c in d.comparison])
    ax.set_title(f"{ds} (n={int(d.n.iloc[0])})"); ax.set_xlabel("ΔAUC (points) · 95% paired-bootstrap CI"); ax.grid(axis="x", color=GRID)
np.atleast_1d(axes)[0].plot([], [], "x", color="#2a78d6", label="paper Table VI (CDF)"); np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig(OUT_DIR / "paired_tests_forest.png", dpi=150); plt.show()

a = SIG[SIG.ablation]
print(f"\nablation ที่ต่างจริงหลัง Holm: {int((a['สรุป'] == 'ต่างจริง').sum())} / {len(a)} การเปรียบเทียบ")
cdf = SIG[(SIG.dataset == "CDF") & SIG.ablation]
if len(cdf):
    print(f"MDE80 บน Celeb-DF เฉลี่ย {cdf.MDE80.mean():.2f} จุด · ผลต่างใน Table VI ของบทความ: "
          f"{cdf['paper_ΔCDF'].abs().min():.2f}–{cdf['paper_ΔCDF'].abs().max():.2f} จุด")

## 2. ความแปรปรวนจาก seed (checkpoint จาก 3d2)

ถ้าไม่มี `ablation_seeds/` ใน Add Input ทุก cell ในส่วนนี้จะไม่โหลดข้อมูลและไม่รันโมเดล

- ประเมิน checkpoint ใหม่ด้วยโค้ดเดียวกับ 3e (main branch, 8 frame, AMP) บนชุดเดียวกัน
- seed 1234 ใช้ score เดิมจาก 3e_V2

In [ ]:
BEIT_CONFIG = dict(drop_path_rate=0.1, use_mean_pooling=False, init_values=0.1, qkv_bias=True,
                   use_abs_pos_emb=False, use_rel_pos_bias=True, use_shared_rel_pos_bias=False)

def make_beit():
    m = modeling_finetune.beit_base_patch16_224(**BEIT_CONFIG)
    m.head = nn.Identity()
    return m

def evidential_uncertainty(logits, K=2):
    alpha = torch.exp(torch.clamp(logits, -10, 10)) + 1
    return K / alpha.sum(dim=1, keepdim=True)

class Detector(nn.Module):
    def __init__(self, dual: bool, ufm: bool = True):
        super().__init__()
        self.dual, self.ufm = dual, ufm
        self.backbone_model = make_beit()
        if dual: self.backbone_model2 = make_beit()
        self.norm_for_cls = nn.LayerNorm(768)
        self.header = nn.Sequential(nn.Linear(768, 2))
    def classify(self, f): return self.header(self.norm_for_cls(f))
    def forward(self, x):
        f1 = self.backbone_model(x)
        out = {"main": self.classify(f1)}
        if self.dual:
            f2 = self.backbone_model2(x)
            l2 = self.classify(f2)
            if self.ufm:
                u = torch.cat([evidential_uncertainty(out["main"].float()),
                               evidential_uncertainty(l2.float())], dim=-1)
                w = torch.softmax(-10 * u, dim=-1).to(f1.dtype)
                fused = w[:, :1] * f1 + w[:, 1:] * f2
            else:
                fused = f1 + f2
            out["branch2"] = l2
            out["fused"] = self.classify(fused)
        return out

def load_state_dict_any(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = {k: v for k, v in ck.items() if not isinstance(v, (dict, torch.Tensor))} if isinstance(ck, dict) else {}
    flags = ck.get("flags") if isinstance(ck, dict) else None
    if flags is None and isinstance(ck, dict) and isinstance(ck.get("cfg"), dict):
        flags = ck["cfg"].get("flags")
    sd = ck
    for key in ("state_dict", "model", "model_state_dict", "net"):
        if isinstance(sd, dict) and isinstance(sd.get(key), dict):
            sd = sd[key]; break
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items() if torch.is_tensor(v)}
    return sd, meta, flags

def load_detector(path):
    sd, meta, flags = load_state_dict_any(path)
    dual = any(k.startswith("backbone_model2.") for k in sd)
    ufm = True if flags is None else bool(flags.get("ufm", True))
    m = Detector(dual=dual, ufm=ufm)
    expected = set(m.state_dict())
    loadable = {k: v for k, v in sd.items() if k in expected}
    missing = sorted(expected - set(loadable))
    bad = [k for k, v in loadable.items() if v.shape != m.state_dict()[k].shape]
    if missing or bad:
        raise RuntimeError(f"{path}: checkpoint ไม่ตรงกับโมเดล missing={missing[:5]} shape={bad[:5]}")
    m.load_state_dict(loadable, strict=True)
    info = {"dual": dual, "ufm": ufm, "flags": flags, "epoch": meta.get("epoch"), "variant": meta.get("variant")}
    return m.eval().to(DEVICE), info


MODELS = [(f"ours_{p.parent.name}", p) for p in SEED_CKPTS]
for n, p in MODELS:
    print(f"  {n:<24} {p}  ({p.stat().st_size/1e6:.0f} MB)")

In [ ]:
# ---------- FF++ test (จากระดับ 3a) ----------
FF_METHODS = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
REAL_REL = "original_sequences/youtube/c23/videos"

def frames_of(d, k=N_FRAMES):
    """เหมือน get_sampled_idx() ของ repo: step = N // k แล้วเอา k ตัวแรก"""
    if not d.is_dir(): return []
    idxs = sorted(int(p.stem) for p in d.glob("*.png") if p.stem.isdigit())
    if not idxs: return []
    sel = idxs[::len(idxs) // k][:k] if len(idxs) >= k else sorted(np.resize(np.array(idxs), k).tolist())
    return [str(d / f"{i}.png") for i in sorted(sel)]

def build_ffpp_test():
    tars = sorted(INPUT_DIR.rglob("*.tar")) if INPUT_DIR.exists() else []
    tars = {t.stem: t for t in tars if t.stem in ["meta", "original"] + FF_METHODS}
    face_root = None
    if "meta" in tars:
        with tarfile.open(tars["meta"]) as tf: tf.extractall(DATA_ROOT)
        face_root = DATA_ROOT / "FF++" / "face_v1"
    else:
        hits = sorted(INPUT_DIR.rglob("face_v1/splits/test.json")) if INPUT_DIR.exists() else []
        if hits: face_root = hits[0].parent.parent
    if face_root is None or not (face_root / "splits" / "test.json").exists():
        raise RuntimeError("ไม่พบข้อมูล FF++ จากระดับ 3a — Add Input เป็น output ของ notebook FFD_level3a")
    pairs = json.load(open(face_root / "splits" / "test.json"))
    rows = []
    for t, s in pairs:
        rows += [{"rel": f"{REAL_REL}/{x}", "label": 0, "method": "original", "video": x} for x in (t, s)]
        for m in FF_METHODS:
            rows += [{"rel": f"manipulated_sequences/{m}/c23/videos/{x}", "label": 1, "method": m, "video": x}
                     for x in (f"{t}_{s}", f"{s}_{t}")]
    # แตกเฉพาะไฟล์ของ test split จาก tar (ประหยัดเวลา/พื้นที่)
    want = tuple(f"FF++/face_v1/{r['rel']}/" for r in rows)
    for stem in ["original"] + FF_METHODS:
        if stem not in tars: continue
        marker = DATA_ROOT / f".test_{stem}"
        if marker.exists(): continue
        with tarfile.open(tars[stem]) as tf:
            tf.extractall(DATA_ROOT, members=[m for m in tf.getmembers() if m.name.startswith(want)])
        marker.touch()
    out = []
    for r in rows:
        fr = frames_of(face_root / r["rel"] / "frame")
        if len(fr) == N_FRAMES:
            out.append({"uid": f"{r['method']}/{r['video']}", "label": r["label"], "group": r["method"], "frames": fr})
    df = pd.DataFrame(out)
    print(f"FF++ test: {len(df)} วิดีโอ (คาด 700) · real {int((df.label==0).sum())} / fake {int((df.label==1).sum())}")
    return df

# ---------- Celeb-DF (crop จากระดับ 3c) ----------
def build_cdf():
    roots = sorted(p for p in INPUT_DIR.rglob("cdf_crops") if p.is_dir()) if INPUT_DIR.exists() else []
    if not roots:
        raise RuntimeError("ไม่พบ cdf_crops — Add Input เป็น output ของระดับ 3c")
    root, out = roots[0], []
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        subset = d.name.split("__")[0]
        fr = sorted(d.glob("*.png"), key=lambda p: int(p.stem) if p.stem.isdigit() else 0)
        if not fr: continue
        fr = [str(p) for p in (fr[:N_FRAMES] if len(fr) >= N_FRAMES else np.resize(np.array(fr), N_FRAMES))]
        out.append({"uid": d.name, "label": int("synthesis" in subset.lower()), "group": subset, "frames": fr})
    df = pd.DataFrame(out)
    print(f"Celeb-DF: {len(df)} วิดีโอ (คาด 518) · real {int((df.label==0).sum())} / fake {int((df.label==1).sum())}  ← {root}")
    return df

DATA = {}
if "FF++" in DATASETS: DATA["FF++"] = build_ffpp_test()
if "CDF"  in DATASETS: DATA["CDF"]  = build_cdf()

In [ ]:
# ---------- face detector + align (เหมือนระดับ 3c) ----------
detector = FaceDetector(0 if DEVICE.type == "cuda" else -1)

def crop_face_test(img, bbox):
    H, W = img.shape[:2]
    (x0, y0), (x1, y1) = bbox
    w, h = x1 - x0, y1 - y0
    w0 = w / 4 * 0.5; h0 = h / 4 * 0.5
    return img[max(0, int(y0-h0)):min(H, int(y1+h0)+1), max(0, int(x0-w0)):min(W, int(x1+w0)+1)]

def realign(img_bgr):
    """คืน (ภาพ 224x224, aligned?) — ขยายภาพก่อนตรวจ เพราะภาพ WDF เป็นหน้าที่ crop มาแล้วขนาดเล็ก"""
    big = cv2.resize(img_bgr, (448, 448), interpolation=cv2.INTER_LINEAR)
    try:
        faces = [f for f in detector.detect(big) if f[2] >= DET_THRES]
    except Exception:
        faces = []
    if faces:
        box, lm, _ = max(faces, key=lambda f: (f[0][2]-f[0][0]) * (f[0][3]-f[0][1]))
        try:
            aligned, _, bb = face_utils.crop_aligned(big, lm.astype(np.float64), landmarks_68=None,
                                                    bboxes=box.astype(np.float64), aligned_image_size=IMG_SIZE)
            crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
            if crop.size:
                return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR), True
        except Exception:
            pass
    return cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR), False

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")
num_key = lambda n: (int(re.sub(r"\D", "", Path(n).stem) or 0), n)

def pick_k(names, k=N_FRAMES):
    names = sorted(names, key=num_key)
    if len(names) >= k:
        return names[::len(names) // k][:k]
    return sorted(np.resize(np.array(names), k).tolist(), key=num_key)

def build_wdf():
    meta_csv = WDF_DIR / "wdf_meta.csv"
    prev = sorted(INPUT_DIR.rglob("wdf_crops/wdf_meta.csv")) if INPUT_DIR.exists() else []
    if prev and not meta_csv.exists():
        print("ใช้ wdf_crops จาก Add Input:", prev[0].parent)
        m = pd.read_csv(prev[0]); root = prev[0].parent
        m["frames"] = m["frames"].map(lambda s: [str(root / x) for x in json.loads(s)])
        return m
    from huggingface_hub import HfApi, hf_hub_download
    files = HfApi().list_repo_files(WDF_REPO, repo_type="dataset")
    todo = []
    for split, lab in WDF_SPLITS.items():
        fs = sorted([f for f in files if f"/{split}/" in f and f.endswith(".tar.gz")], key=num_key)
        todo += [(f, split, lab) for f in fs[:WDF_MAX_FILES]]
    print(f"WildDeepfake: {len(todo)} ไฟล์ tar · " +
          ", ".join(f"{s} {sum(t[1]==s for t in todo)}" for s in WDF_SPLITS))
    rows, n_aligned, n_img, shown = [], 0, 0, False
    dl_dir = TMP_DIR / "wdf_dl"; dl_dir.mkdir(parents=True, exist_ok=True)
    for f, split, lab in tqdm(todo, desc="WDF"):
        local = Path(hf_hub_download(WDF_REPO, f, repo_type="dataset", local_dir=str(dl_dir)))
        vid = Path(f).name.replace(".tar.gz", "")
        with tarfile.open(local, "r:*") as tf:
            names = [m.name for m in tf.getmembers() if m.isfile() and m.name.lower().endswith(IMG_EXT)]
        seqs = {}
        for n in names: seqs.setdefault(str(Path(n).parent), []).append(n)
        if not shown:
            print(f"\nตัวอย่างโครงสร้าง {f}: {len(names)} ภาพ, {len(seqs)} sequence")
            for s_, ns in list(seqs.items())[:3]: print("  ", s_, "→", len(ns), "ภาพ เช่น", ns[:2])
            shown = True
        want = {}
        for s_, ns in seqs.items():
            for j, n in enumerate(pick_k(ns)): want.setdefault(n, []).append((s_, j))
        imgs = {}
        with tarfile.open(local, "r|*") as tf:               # อ่านรอบเดียวแบบ stream (r|* = ตรวจเองว่าบีบอัดหรือไม่ — ไฟล์ .tar.gz ของ WDF เป็น tar ธรรมดา)
            for m in tf:
                if m.name in want:
                    buf = np.frombuffer(tf.extractfile(m).read(), np.uint8)
                    imgs[m.name] = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        for s_, ns in seqs.items():
            sid = re.sub(r"[^\w.-]", "_", f"{vid}__{s_}")
            out_dir = WDF_DIR / split / sid; out_dir.mkdir(parents=True, exist_ok=True)
            rel = []
            for j, n in enumerate(pick_k(ns)):
                im = imgs.get(n)
                if im is None: continue
                if WDF_REALIGN:
                    im, ok = realign(im); n_aligned += ok
                else:
                    im = cv2.resize(im, (IMG_SIZE, IMG_SIZE))
                n_img += 1
                p = out_dir / f"{j}.png"; cv2.imwrite(str(p), im); rel.append(str(p.relative_to(WDF_DIR)))
            if rel:
                rel = (rel * N_FRAMES)[:N_FRAMES]
                rows.append({"uid": f"{split}/{sid}", "label": lab, "group": split, "source_video": vid,
                             "n_frames_seq": len(ns), "frames": json.dumps(rel)})
        local.unlink(missing_ok=True)
    m = pd.DataFrame(rows)
    m.to_csv(meta_csv, index=False)
    print(f"\nrealign สำเร็จ {n_aligned}/{n_img} ภาพ ({100*n_aligned/max(n_img,1):.1f}%) — ที่เหลือใช้ภาพเดิม resize เป็น 224")
    m["frames"] = m["frames"].map(lambda s: [str(WDF_DIR / x) for x in json.loads(s)])
    return m

if "WDF" in DATASETS:
    t0 = time.time()
    DATA["WDF"] = build_wdf()
    w = DATA["WDF"]
    print(f"WildDeepfake: {len(w)} sequence จาก {w['source_video'].nunique()} วิดีโอ · "
          f"real {int((w.label==0).sum())} / fake {int((w.label==1).sum())} · {(time.time()-t0)/60:.1f} นาที")

In [ ]:
def to_tensor(rgb):
    x = (rgb.astype(np.float32) / 255.0 - np.array(MEAN, np.float32)) / np.array(STD, np.float32)
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

class FrameList(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        rgb = np.asarray(Image.open(self.paths[i]).convert("RGB"))
        if rgb.shape[:2] != (IMG_SIZE, IMG_SIZE):
            rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return to_tensor(rgb)

@torch.no_grad()
def score_dataset(model, df):
    paths = [p for fs in df["frames"] for p in fs]
    owner = np.repeat(np.arange(len(df)), [len(fs) for fs in df["frames"]])
    loader = DataLoader(FrameList(paths), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    acc = {}
    for x in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP and DEVICE.type == "cuda"):
            out = model(x)
        for h, lg in out.items():
            lg = lg.float()
            acc.setdefault(f"p_{h}", []).append(torch.softmax(lg, 1)[:, 1].cpu().numpy())
            acc.setdefault(f"u_{h}", []).append(evidential_uncertainty(lg)[:, 0].cpu().numpy())
    fr = pd.DataFrame({k: np.concatenate(v) for k, v in acc.items()}); fr["row"] = owner
    vid = fr.groupby("row").mean()                                  # เฉลี่ย 8 frame = score วิดีโอ
    return pd.concat([df[["uid", "label", "group"]].reset_index(drop=True), vid.reset_index(drop=True)], axis=1)

SCORES, INFO = {}, {}
for name, path in MODELS:
    t0 = time.time()
    model, info = load_detector(path); INFO[name] = info
    for ds, df in DATA.items():
        v = score_dataset(model, df)
        v.to_csv(OUT_DIR / f"scores_{name}_{ds}.csv", index=False)
        SCORES[(name, ds)] = v
    print(f"{name:<20} dual={info['dual']!s:<5} ufm={info['ufm']!s:<5} · {(time.time()-t0)/60:.1f} นาที")
    del model; torch.cuda.empty_cache()

for k, v in SCORES.items():
    S[k] = v

In [ ]:
if SEED_CKPTS:
    rx = re.compile(r"ours_(.+)_s(\d+)$")
    runs = []
    for (nm, ds) in S:
        m = rx.match(nm)
        if m: runs.append((m.group(1), int(m.group(2)), nm, ds))
    for v in sorted({r[0] for r in runs}):
        for ds in DS_P1:
            if (vname(v), ds) in S: runs.append((v, 1234, vname(v), ds))
    R = pd.DataFrame([{"variant": v, "seed": sd, "dataset": ds,
                       "AUC": 100 * roc_auc_score(S[(nm, ds)].label, S[(nm, ds)].p_main)} for v, sd, nm, ds in runs])
    R = R.drop_duplicates(["variant", "seed", "dataset"])
    W = R.pivot_table(index=["variant", "seed"], columns="dataset", values="AUC").round(2)
    print("AUC (%) ต่อ seed · main branch"); print(W.to_string())
    agg = R.groupby(["variant", "dataset"]).AUC.agg(["mean", "std", "min", "max", "count"]).round(2)
    print("\nmean ± SD ข้าม seed"); print(agg.to_string())
    R.to_csv(OUT_DIR / "seed_auc_long.csv", index=False); agg.to_csv(OUT_DIR / "seed_auc_summary.csv")

    vs = sorted(R.variant.unique())
    if {"fb_ce", "cfb_full"} <= set(vs):
        print("\nΔAUC (cfb_full − fb_ce) ต่อ seed  · บทความ (CDF): +1.28")
        D = R.pivot_table(index=["seed", "dataset"], columns="variant", values="AUC")
        D["Δ (full − ce)"] = D["cfb_full"] - D["fb_ce"]
        print(D.round(2).to_string())
        # paired DeLong บน score เฉลี่ยข้าม seed (seed ensemble) — ลดความแปรปรวนจากการฝึก
        ens_rows = []
        for ds in DS_P1:
            seeds = sorted(set(R[(R.dataset == ds) & (R.variant == "fb_ce")].seed) & set(R[(R.dataset == ds) & (R.variant == "cfb_full")].seed))
            if len(seeds) < 2: continue
            nm = lambda v, sd: vname(v) if sd == 1234 else f"ours_{v}_s{sd}"
            names = [nm(v, sd) for v in ["fb_ce", "cfb_full"] for sd in seeds]
            y, T = aligned(ds, names)
            pa = T[[nm("fb_ce", sd) for sd in seeds]].mean(1).values; pb = T[[nm("cfb_full", sd) for sd in seeds]].mean(1).values
            aA, aB, z, p = delong(y, pa, pb)
            ens_rows.append({"dataset": ds, "n_seeds": len(seeds), "AUC fb_ce (ens)": 100 * aA,
                             "AUC cfb_full (ens)": 100 * aB, "ΔAUC": 100 * (aB - aA), "p_DeLong": p})
        if ens_rows:
            ENS = pd.DataFrame(ens_rows); ENS.to_csv(OUT_DIR / "seed_ensemble_test.csv", index=False)
            print("\nseed ensemble (เฉลี่ย p ทุก seed) · paired DeLong"); print(ENS.round(4).to_string(index=False))
else:
    print("ไม่มี checkpoint จาก 3d2 — ข้ามส่วน 2")

## 3. สรุป

In [ ]:
print("=" * 72); print("สรุประดับ 3f"); print("=" * 72)
for ds in DS_P1:
    d = SIG[(SIG.dataset == ds)]
    print(f"\n{ds}:")
    for _, r in d.iterrows():
        ph = f"p_Holm={r.p_Holm:.3f}" if r.ablation else f"p={r.p_DeLong:.2g}"
        print(f"  {r.comparison:<38} Δ = {r['ΔAUC']:+6.2f}  [{r.CI_lo:+.2f}, {r.CI_hi:+.2f}]  {ph:<14} → {r['สรุป']}")
print("\nไฟล์ผลลัพธ์:", sorted(p.name for p in OUT_DIR.iterdir()))
print(f"เวลารวม: {(time.time() - T_START) / 60:.1f} นาที")